# 📊 RAG Chatbot — PDF + LLM
### Step-by-step notebook: Load PDF → Build Vector Store → Ask Questions
**Fixes applied:** full context (no truncation), k=3, max_tokens=300

### Step 1: Install Packages (Run Once)

In [1]:
# Run this cell once to install all required packages
# !pip install openai langchain langchain-community langchain-text-splitters pypdf faiss-cpu sentence-transformers -q

### Step 2: Import Libraries

In [2]:
import warnings
import os
import time

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print('✅ All imports successful!')

✅ All imports successful!


### Step 3: Configure API Settings

In [3]:
# ── Change these to match your setup ──────────────────────
API_KEY  = 'ollama'                      # use 'ollama' for local
BASE_URL = 'http://localhost:11434/v1'   # Ollama local URL
MODEL    = 'phi3'                        # your Ollama model name
PDF_FILE = 'business_report.pdf'         # path to your PDF
# ──────────────────────────────────────────────────────────

print(f'✅ Config ready')
print(f'   Model   : {MODEL}')
print(f'   Base URL: {BASE_URL}')
print(f'   PDF     : {PDF_FILE}')

✅ Config ready
   Model   : phi3
   Base URL: http://localhost:11434/v1
   PDF     : business_report.pdf


### Step 4: Test API Connection

In [4]:
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

try:
    print('🧪 Testing connection...')
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'Reply with: Working!'}],
        max_tokens=10
    )
    print(f'✅ API OK: {response.choices[0].message.content}')
except Exception as e:
    print(f'❌ API Error: {e}')
    print('   Fix: Make sure Ollama is running → terminal: ollama serve')

🧪 Testing connection...
❌ API Error: Connection error.
   Fix: Make sure Ollama is running → terminal: ollama serve


### Step 5: Load PDF Document

In [5]:
print(f'📄 Loading PDF: {PDF_FILE}')

try:
    loader = PyPDFLoader(PDF_FILE)
    docs = loader.load()
    print(f'✅ Loaded {len(docs)} pages')
    print(f'   Preview of page 1 (first 200 chars):')
    print(f'   {docs[0].page_content[:200]}')
except Exception as e:
    print(f'❌ Error loading PDF: {e}')
    print('   Check: file exists in same folder, correct name, valid PDF')

📄 Loading PDF: business_report.pdf
✅ Loaded 10 pages
   Preview of page 1 (first 200 chars):
   Meta Reports Fourth Quarter and Full Year 2024 Results
MENLO PARK, Calif. – January 29, 2025  – Meta Platforms, Inc. (Nasdaq: META) today reported financial results for the 
quarter and full year ende


### Step 6: Split into Chunks

In [6]:
# chunk_size=500  → each chunk is max 500 characters
# chunk_overlap=50 → chunks share 50 chars (no info lost at edges)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = splitter.split_documents(docs)

print(f'✅ Document split into {len(splits)} chunks')
print(f'   Pages : {len(docs)}')
print(f'   Chunks: {len(splits)}')
print(f'\n   Sample chunk 2 (first 300 chars):')
print(f'   {splits[1].page_content[:300]}')

✅ Document split into 51 chunks
   Pages : 10
   Chunks: 51

   Sample chunk 2 (first 300 chars):
   Three Months Ended December 31,
 % Change
Twelve Months Ended December 31,
% Change
In millions, except percentages and 
per share amounts 2024 2023 2024 2023
Revenue $ 48,385 $ 40,111  21 % $ 164,501 $ 134,902  22 %
Costs and expenses  25,020  23,727  5 %  95,121  88,151  8 %
Income from operations


### Step 7: Create Embeddings + Vector Store (FAISS)

In [7]:
print('⏳ Creating embeddings (30-60 seconds on first run)...')
print('   Note: warnings below are normal, ignore them\n')

start = time.time()

# Convert text chunks → numbers (embeddings)
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-MiniLM-L3-v2'
)

# Store all embeddings in FAISS vector database
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)

elapsed = time.time() - start
print(f'✅ Vector store created in {elapsed:.1f} seconds!')
print(f'   {len(splits)} chunks are now searchable')

⏳ Creating embeddings (30-60 seconds on first run)...
   Note: warnings below are normal, ignore them



Loading weights: 100%|██████████| 55/55 [00:00<00:00, 187.91it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store created in 112.4 seconds!
   51 chunks are now searchable


### Step 8: Define Helper Functions

In [8]:
def get_completion(prompt):
    """Call the LLM and get a response."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=300
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"


def build_prompt(query, docs):
    """Build precise prompt — finds ACTUAL figures, not forecasts."""
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""You are a precise financial analyst assistant.
Answer the question using ONLY the PDF context below.

CRITICAL RULES:
1. Always report ACTUAL reported figures, NOT forecasts or future guidance.
2. Example: "capital expenditures 2024" = find the ACTUAL 2024 number reported.
   Do NOT confuse with 2025 forecast/guidance numbers.
3. State financial figures EXACTLY as they appear in the context.
4. If information is not found, say: "This information is not in the document."
5. Be concise and precise.

CONTEXT FROM PDF:
{context}

QUESTION: {query}

ANSWER:"""
    return prompt


def ask_question(query, k=3, show_progress=True):
    """Full RAG pipeline: Retrieve -> Augment -> Generate."""
    if show_progress:
        print(f"   [1/3] Searching PDF for top {k} chunks...")

    docs_with_scores = vectorstore.similarity_search_with_score(query, k=k)

    if show_progress:
        print("   [2/3] Building prompt with full context...")

    docs = [doc for doc, score in docs_with_scores]
    prompt = build_prompt(query, docs)

    if show_progress:
        print(f"   [3/3] Asking LLM... (prompt: {len(prompt)} chars)")

    response = get_completion(prompt)
    return response, docs_with_scores


print("Helper functions ready!")
print("   ask_question(query, k=3) — use this to ask questions")


Helper functions ready!
   ask_question(query, k=3) — use this to ask questions


### Step 9: Test with Sample Questions

In [9]:
test_questions = [
    'What was Meta total revenue in 2024?',
    'Who is the CEO of Meta?',
    'What was the net income for full year 2024?',
    'What was the Q4 2024 revenue?',
]

print('🧪 Testing RAG System\n')
print('='*65)

for query in test_questions:
    print(f'\n❓ Question: {query}')
    print('-'*65)

    start = time.time()
    response, docs = ask_question(query, k=3)
    elapsed = time.time() - start

    print(f'\n💬 Answer: {response}')
    print(f'⏱️  Time: {elapsed:.1f}s')
    print('='*65)

🧪 Testing RAG System


❓ Question: What was Meta total revenue in 2024?
-----------------------------------------------------------------
   [1/3] Searching PDF for top 3 chunks...
   [2/3] Building prompt with full context...
   [3/3] Asking LLM... (prompt: 2011 chars)

💬 Answer: Error: Connection error.
⏱️  Time: 13.7s

❓ Question: Who is the CEO of Meta?
-----------------------------------------------------------------
   [1/3] Searching PDF for top 3 chunks...
   [2/3] Building prompt with full context...
   [3/3] Asking LLM... (prompt: 1929 chars)

💬 Answer: Error: Connection error.
⏱️  Time: 13.5s

❓ Question: What was the net income for full year 2024?
-----------------------------------------------------------------
   [1/3] Searching PDF for top 3 chunks...
   [2/3] Building prompt with full context...
   [3/3] Asking LLM... (prompt: 1860 chars)

💬 Answer: Error: Connection error.
⏱️  Time: 13.5s

❓ Question: What was the Q4 2024 revenue?
--------------------------------------

### Step 10: Ask Your Own Question

In [10]:
# ── Change this question to anything you want ──
my_question = 'What was Meta total revenue in 2024?'
# ───────────────────────────────────────────────

print(f'❓ Question: {my_question}\n')

start = time.time()
response, docs = ask_question(my_question, k=3)
elapsed = time.time() - start

print(f'\n💬 Answer: {response}')
print(f'⏱️  Time: {elapsed:.1f}s')

❓ Question: What was Meta total revenue in 2024?

   [1/3] Searching PDF for top 3 chunks...
   [2/3] Building prompt with full context...
   [3/3] Asking LLM... (prompt: 2011 chars)

💬 Answer: Error: Connection error.
⏱️  Time: 13.5s


### Step 11: Ask Multiple Questions at Once

In [11]:
# ── Add your questions here ──
my_questions = [
    'What is the total revenue for full year 2024?',
    'What is the net income for full year 2024?',
    'Who are the key people mentioned?',
    'What are the capital expenditures in 2024?',
    # Add more questions below...
]
# ─────────────────────────────

print('🔄 Processing all questions...\n')

for i, query in enumerate(my_questions, 1):
    print(f'[{i}/{len(my_questions)}] ❓ {query}')

    start = time.time()
    response, _ = ask_question(query, k=3, show_progress=False)
    elapsed = time.time() - start

    print(f'💬 {response}')
    print(f'⏱️  {elapsed:.1f}s\n')

print('✅ All done!')

🔄 Processing all questions...

[1/4] ❓ What is the total revenue for full year 2024?
💬 Error: Connection error.
⏱️  13.5s

[2/4] ❓ What is the net income for full year 2024?
💬 Error: Connection error.
⏱️  13.5s

[3/4] ❓ Who are the key people mentioned?
💬 Mark Zuckerberg and Sheryl Sandberg (as implied by her role as COO) are likely to be the key individuals referenced in this context. However, specific titles or roles for other executives aren't provided within the given text snippet from the PDF document. To confirm their involvement with Meta/Facebook specifically regarding disclosure obligations and material non-public information would require additional details not included herein.
⏱️  135.3s

[4/4] ❓ What are the capital expenditures in 2024?
💬 Capital expenditures were reported to be $39.23 billion for the full year of 2024.
⏱️  43.9s

✅ All done!


### Step 12: Debug — View Retrieved Chunks

In [12]:
# See which chunks were retrieved for any question
debug_query = 'What was Meta total revenue in 2024?'

print(f'🔍 Debug: chunks retrieved for:')
print(f'   "{debug_query}"\n')

_, docs_with_scores = ask_question(debug_query, k=3, show_progress=False)

for i, (doc, score) in enumerate(docs_with_scores):
    print(f'--- Chunk {i+1} | Similarity Score: {score:.4f} ---')
    print(doc.page_content)
    print(f'   Has $164 billion: {"164" in doc.page_content}')
    print()

🔍 Debug: chunks retrieved for:
   "What was Meta total revenue in 2024?"

--- Chunk 1 | Similarity Score: 13.3684 ---
META PLATFORMS, INC.
CONDENSED CONSOLIDATED STATEMENTS OF INCOME
(In millions, except per share amounts)
(Unaudited)
Three Months Ended December 31, Twelve Months Ended December 31,
2024 2023 2024 2023
Revenue $ 48,385 $ 40,111 $ 164,501 $ 134,902 
Costs and expenses:  
Cost of revenue  8,839  7,695  30,161  25,959 
Research and development  12,180  10,517  43,873  38,483 
Marketing and sales  3,240  3,226  11,347  12,301 
General and administrative (1)  761  2,289  9,740  11,408
   Has $164 billion: True

--- Chunk 2 | Similarity Score: 14.0058 ---
Meta Reports Fourth Quarter and Full Year 2024 Results
MENLO PARK, Calif. – January 29, 2025  – Meta Platforms, Inc. (Nasdaq: META) today reported financial results for the 
quarter and full year ended December 31, 2024.
"We continue to make good progress on AI, glasses, and the future of social media," said Mark Zuckerberg,

---
## ✅ Summary of Fixes Applied

| Bug | Old Code | Fixed Code |
|-----|----------|------------|
| Context truncated | `doc.page_content[:500]` | `doc.page_content` (full) |
| Too few chunks | `k=1` | `k=3` |
| Short answers | No `max_tokens` | `max_tokens=300` |
| Wrong prompt | Generic prompt | Financial analyst prompt |

## 💡 Tips
- Use `k=3` for accurate financial answers (recommended)
- Use `k=5` for complex multi-part questions
- Always check Step 12 debug if you get wrong answers